# 01 — Activity and BPI review

Review the independent response/BPI computation before any canonical export. Whole-population ROI results and HCR-identified-cell results stay separate.

In [ ]:
# 02 — Choose the fish and saved run
from pathlib import Path
from codeants_2pf_hcr.qc_notebooks import QCNotebookConfig, resolve_qc_notebook_context

FISH_ID = "L765_f04"
LOCAL_ROOT = Path("/Volumes/dataDrive/dataProcessing/2p_processing")
PIPELINE_ROOT = Path("/Volumes/dataDrive/dataProcessing/2p_processing/_staged_pipeline_runs/20260803T112035Z_functional_registration_ants_no_fallback/L765_f04")
CONTEXT = resolve_qc_notebook_context(QCNotebookConfig(FISH_ID, LOCAL_ROOT, PIPELINE_ROOT))
MANIFEST_ROOT = CONTEXT.fish_dir / "03_analysis" / "functional" / "pipeline_manifests"


In [ ]:
# 03 — Locate the independent activity/BPI outputs and optional identity context
REGISTRATION_DIR = CONTEXT.stage_paths["score-activity-bpi"] / "registration"
ROI_MASTER = REGISTRATION_DIR / "functional_roi_activity_identity.csv"
BPI_CELLS = REGISTRATION_DIR / "functional_roi_activity_bpi_cells.csv"
IDENTITY_DIR = CONTEXT.stage_paths["assign-hcr-identity"] / "registration"
HCR_STATUS = IDENTITY_DIR / "hcr_activity_status.csv"
HCR_PAIRS = IDENTITY_DIR / "conf_to_func_pairs.csv"
TRACE_META = CONTEXT.fish_dir / "03_analysis" / "functional" / "derived" / "suite2p_traces" / "suite2p_dff_traces_meta.csv"
EXPECTED_FIGURES = []  # Figures are reviewed only after an explicit export/figure stage.
MIDLINE_REVIEW_DIR = CONTEXT.stage_paths["register-functional-to-anatomy"] / "reviews"
MIDLINE_ACCEPTED_CANDIDATES = sorted(MIDLINE_REVIEW_DIR.glob("midline_annotation_*.json"))
ACCEPTED_MIDLINE_SIDECAR = MIDLINE_ACCEPTED_CANDIDATES[0] if len(MIDLINE_ACCEPTED_CANDIDATES) == 1 else None
if len(MIDLINE_ACCEPTED_CANDIDATES) != 1:
    display({"accepted_midline_sidecar": "select exactly one timestamped Notebook 01 sidecar before laterality review", "candidates": [str(path) for path in MIDLINE_ACCEPTED_CANDIDATES]})


In [ ]:
# 04 — Summarize independent activity/BPI results
from IPython.display import display
from codeants_2pf_hcr.plots.qc_molecular import inspect_activity_export_qc, plot_activity_export_qc, plot_activity_bpi_gate

report = inspect_activity_export_qc(
    fish_id=FISH_ID,
    pipeline_root=CONTEXT.pipeline_root,
    fish_dir=CONTEXT.fish_dir,
    roi_master_path=ROI_MASTER,
    bpi_cells_path=BPI_CELLS,
    hcr_status_path=HCR_STATUS,
    hcr_pairs_path=HCR_PAIRS,
    trace_meta_path=TRACE_META,
    require_hcr_exports=False, require_trace_meta=False,
    expected_figure_paths=EXPECTED_FIGURES,
)
display(report["issues"], report["scope_summary"], report["response_summary"], report["figures"])
plot_activity_export_qc(report);


## 05 — Decide whether the activity/BPI results are ready

Check both session logs, ROI table keys, response/BPI counts, and the separate ROI and HCR result sets. Missing identity or figure outputs remain visible but do not change the independent activity calculation; this notebook does not create or accept results.

In [ ]:
# 09 — Q7 global laterality AUC (read-only; fail closed until the Q2 sidecar is accepted)
from codeants_2pf_hcr.plots.analysis import render_accepted_global_laterality_auc_qc

if ACCEPTED_MIDLINE_SIDECAR is None:
    display({'global_laterality_auc': 'blocked', 'reason': 'select exactly one accepted timestamped Notebook 01 midline sidecar in Cell 03'})
else:
    MOTION_AUC_POINTS = REGISTRATION_DIR / 'motion_auc_plot_points.csv'
    MOTION_AUC_COUNTS = REGISTRATION_DIR / 'motion_auc_plot_counts.csv'
    GLOBAL_LATERALITY_AUC = render_accepted_global_laterality_auc_qc(
        fish_id=FISH_ID, master_roi_csv=ROI_MASTER, points_csv=MOTION_AUC_POINTS,
        counts_csv=MOTION_AUC_COUNTS, accepted_midline_sidecar=ACCEPTED_MIDLINE_SIDECAR,
    )
    display(GLOBAL_LATERALITY_AUC['fig'])
    print('[Q7 global laterality AUC]', GLOBAL_LATERALITY_AUC['status'])
    if GLOBAL_LATERALITY_AUC['status'] == 'blocked':
        print(GLOBAL_LATERALITY_AUC['reason'])


## 05 — Inspect response calls and BPI classes

These ROI-centric panels use the cut-offs persisted by `score-activity-bpi`. They do not infer laterality; the global ipsilateral/contralateral AUC view remains blocked pending accepted midline placement.

In [ ]:
# 06 — Render the read-only Q7 activity/BPI gate
BPI_GATE_FIGURE, BPI_GATE_ROWS = plot_activity_bpi_gate(BPI_CELLS, fish_id=FISH_ID)
display(BPI_GATE_FIGURE)
display(BPI_GATE_ROWS['bpi_category'].value_counts().rename_axis('bpi_category').rename('n_rois'))


## 07 — Manual-midline laterality consequence review

These are the required read-only `[56f-qc]` views. They use every ROI in the persisted ROI inventory and the exact finite-activity rows from the persisted response-scoring table. A proposed line may be drawn dashed for comparison, but only an accepted, hash-bound anatomy-space annotation determines side colours. Global ipsilateral/contralateral AUC remains blocked here.

In [ ]:
# 08 — Render all-ROI and response-scored laterality grids (read-only)
from codeants_2pf_hcr.plots.qc_functional import load_functional_registration_qc
from codeants_2pf_hcr.plots.qc_midline import build_midline_laterality_qc_tables, render_midline_activity_plane_grid, render_midline_laterality_plane_grid
from codeants_2pf_hcr.spatial import imread_any

if ACCEPTED_MIDLINE_SIDECAR is None:
    display({'laterality_grids': 'blocked', 'reason': 'select exactly one accepted timestamped Notebook 01 midline sidecar in Cell 03'})
else:
    REGISTRATION_QC = load_functional_registration_qc(
        fish_id=FISH_ID, anatomy_preparation_manifest_path=None,
        anatomy_stack_path=CONTEXT.fish_dir / '02_reg' / '00_preprocessing' / '2p_anatomy' / f'{FISH_ID}_anatomy_2P_GCaMP.nrrd',
        registration_manifest_path=MANIFEST_ROOT / 'register-functional-to-anatomy_manifest.json',
        roi_transform_manifest_path=MANIFEST_ROOT / 'transform-functional-rois-to-anatomy_manifest.json',
        qc_manifest_path=MANIFEST_ROOT / 'make-functional-registration-qc_manifest.json',
        registration_root=CONTEXT.stage_paths['register-functional-to-anatomy'],
        transformed_roi_root=CONTEXT.stage_paths['transform-functional-rois-to-anatomy'],
        qc_root=CONTEXT.stage_paths['make-functional-registration-qc'],
    )
    MIDLINE_SIDECAR = ACCEPTED_MIDLINE_SIDECAR  # Accepted sidecar written from Notebook 01 manual-midline review.
    MIDLINE_PROPOSAL_LINES = None  # Optional {plane_idx: {x0, y0, theta_deg}}; diagnostic dashed line only.
    PLANE_KEY = next(key for key in ('plane_index', 'index', 'plane_idx') if key in REGISTRATION_QC['plane_refs'].columns)
    PLANE_Z = {int(row[PLANE_KEY]): int(row['best_z']) for _, row in REGISTRATION_QC['plane_refs'].iterrows()}
    ROI_LABEL_IMAGES = {int(row.plane_index): Path(row.path) for row in REGISTRATION_QC['transformed_labels'].itertuples()}
    ANATOMY_ZYX = imread_any(REGISTRATION_QC['anatomy_stack_path'])
    ALL_ROIS_MIDLINE, ACTIVITY_ROIS_MIDLINE, MIDLINE_META = build_midline_laterality_qc_tables(ROI_MASTER, BPI_CELLS, MIDLINE_SIDECAR, fish_id=FISH_ID)
    ALL_ROI_LATERALITY_FIGURE = render_midline_laterality_plane_grid(ALL_ROIS_MIDLINE, MIDLINE_META['midline_context'], ANATOMY_ZYX, PLANE_Z, proposal_lines=MIDLINE_PROPOSAL_LINES, roi_label_images=ROI_LABEL_IMAGES)
    ACTIVITY_LATERALITY_FIGURE = render_midline_activity_plane_grid(ACTIVITY_ROIS_MIDLINE, MIDLINE_META['midline_context'], ANATOMY_ZYX, PLANE_Z, proposal_lines=MIDLINE_PROPOSAL_LINES)
    display(ALL_ROI_LATERALITY_FIGURE, ACTIVITY_LATERALITY_FIGURE)
    display(ALL_ROIS_MIDLINE.groupby(['plane_idx', 'midline_side']).size().rename('n_all_rois'))
    display(ACTIVITY_ROIS_MIDLINE.groupby(['plane_idx', 'midline_side']).size().rename('n_persisted_response_scored_rois'))
